# Icarus — Hyperspectral Soil CNN Training
**HYPERVIEW2 dataset · Google Colab**

Before running: `Runtime → Change runtime type → T4 GPU`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/icarus'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive directory ready: {DRIVE_DIR}')

## 2. Clone repo

Opening this notebook from GitHub (via colab.research.google.com → File → Open → GitHub) only loads the `.ipynb`. This cell clones the rest of the repo (`train.py`, `model.py`, etc.) into the Colab session.

In [ ]:
import os

%cd /content

# Detect which repo this notebook was opened from
repo_url = None
try:
    # Colab sets this env var when a notebook is opened from GitHub
    colab_nb = os.environ.get('COLAB_NOTEBOOK_URL', '')
    if 'github.com' in colab_nb:
        parts = colab_nb.replace('https://github.com/', '').split('/')
        repo_url = f'https://github.com/{parts[0]}/{parts[1]}.git'
except Exception:
    pass

if repo_url is None:
    repo_url = 'https://github.com/davidshukhin/Icarus.git'  # fallback

print(f'Cloning: {repo_url}')
!git clone {repo_url} icarus
%cd /content/icarus

## 3. Install dependencies

In [ ]:
# Colab already has torch/numpy/scipy/sklearn — only install the extras
!pip install -q spectral rasterio pyyaml h5py
print('Dependencies installed.')

## 4. Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

## 5. Download HYPERVIEW2

Two steps:
1. `eotdl` downloads the STAC catalog (already done above)
2. `train.py --prepare` reads the catalog and downloads the actual patch files + labels

In [ ]:
import os

CATALOG  = '/root/.cache/eotdl/datasets/HYPERVIEW2/catalog.v2.parquet'
DATA_DIR = f'{DRIVE_DIR}/data/hyperview2'

# Download catalog if not already cached
if not os.path.exists(CATALOG):
    !pip install -q eotdl
    !eotdl datasets get HYPERVIEW2 -v 2

# Download actual patch files + train_gt.csv from the catalog
LABELS_FILE = f'{DATA_DIR}/train_gt.csv'
if not os.path.exists(LABELS_FILE):
    !python train.py --prepare --catalog {CATALOG} --data_dir {DATA_DIR}
else:
    print(f'Patches already downloaded at {DATA_DIR}')

In [ ]:
# Quick sanity check — show label distribution
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LABELS_FILE)
print(f'Patches: {len(df)}')
print(df.describe())

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, ['K', 'Mg', 'P2O5', 'pH']):
    if col in df.columns:
        df[col].hist(bins=30, ax=ax)
        ax.set_title(col)
plt.tight_layout()
plt.show()

## 6. Train

In [ ]:
SAVE_PATH = f'{DRIVE_DIR}/best_model.pth'

!python train.py \
    --data_dir {DATA_DIR} \
    --model se \
    --epochs 50 \
    --batch_size 32 \
    --save_path {SAVE_PATH}

## 7. Plot training results

In [ ]:
# Parse loss/AUC from the training output captured above.
# Re-run this cell after training completes.
import re
import matplotlib.pyplot as plt

log_path = 'logs/training.log'  # adjust if your setup writes a log file

epochs, train_loss, val_acc, val_auc = [], [], [], []

if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            m = re.search(
                r'Epoch (\d+).*train loss ([\d.]+).*acc ([\d.]+).*AUC mean ([\d.]+)', line
            )
            if m:
                epochs.append(int(m.group(1)))
                train_loss.append(float(m.group(2)))
                val_acc.append(float(m.group(3)))
                val_auc.append(float(m.group(4)))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, train_loss); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
    ax2.plot(epochs, val_auc, label='AUC'); ax2.plot(epochs, val_acc, label='Acc')
    ax2.set_title('Validation'); ax2.set_xlabel('Epoch'); ax2.legend()
    plt.tight_layout(); plt.show()
else:
    print(f'No log file found at {log_path} — training output was printed to stdout above.')

## 8. Run inference on a sample patch

In [ ]:
import torch
import numpy as np
import sys
sys.path.insert(0, '/content/icarus')

from model import create_model
from configs.constants import HEALTH_LABELS, CONTAMINANT_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load checkpoint
checkpoint = torch.load(SAVE_PATH, map_location=device)
cfg = checkpoint['cfg']

model = create_model(
    cfg.get('model_name', 'se'),
    num_bands=cfg['num_bands'],
    num_classes=cfg['num_classes'],
    num_contaminants=cfg['num_contaminants'],
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Pick a random patch
sample_id = df['patch_id'].iloc[0]
patch = np.load(f'{DATA_DIR}/{sample_id}.npz')
cube  = patch[list(patch.keys())[0]].astype(np.float32)
if cube.ndim == 3 and cube.shape[2] < cube.shape[0]:
    cube = cube.transpose(2, 0, 1)

from scipy.ndimage import zoom
if cube.shape[0] != cfg['num_bands']:
    cube = zoom(cube, (cfg['num_bands'] / cube.shape[0], 1, 1), order=1)
tH, tW = cfg.get('target_size', [64, 64])
cube = zoom(cube, (1, tH / cube.shape[1], tW / cube.shape[2]), order=1)

mean = cube.mean(axis=(1,2), keepdims=True)
std  = cube.std(axis=(1,2),  keepdims=True) + 1e-8
cube = (cube - mean) / std

x = torch.from_numpy(cube).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,C,H,W)

with torch.no_grad():
    pred_health, pred_contam = model(x)

health_class = pred_health.argmax(dim=1).item()
health_conf  = pred_health.softmax(dim=1).max().item()
contam_probs = pred_contam.squeeze().cpu().numpy()

print(f'Patch: {sample_id}')
print(f'Soil health: {HEALTH_LABELS[health_class]} (confidence: {health_conf:.1%})')
print('Contaminant probabilities:')
for name, prob in zip(CONTAMINANT_NAMES, contam_probs):
    print(f'  {name}: {prob:.3f}')